# AgriPrice Advisor
## AI-Driven Food Price Forecasting & Decision-Support Platform for Smallholder Farmers in Uganda

**Group 6 Project**

This notebook documents the data and modelling work behind AgriPrice Advisor: exploring the WFP data, building the monthly panel, evaluating forecasters walk-forward, choosing a method per crop, the sell-or-store logic, and testing the advice on past prices.

**Official user flow (see `AgriPrice_Advisor_User_Flow.pdf`)**
- **Part 1**: Sign-in → location/market → crop & quantity → forecast → sell-vs-store + market comparison → dashboard + explanation → accept advice
- **Part 2**: Arrange storage / transport / buyers with ranked providers, quotations, tracking, ratings

**Target crops**: Maize, Maize flour, Beans, Sorghum, Millet, Cassava flour.

The code lives in `src/` (one module per step) and is shared with the Streamlit app in `app/`. See `README.md` for the full layout. Rebuild every result from the raw CSV with:
```bash
pip install -r requirements.txt
python scripts/run_pipeline.py --notebook
streamlit run app/app.py
```


## 1. Setup & Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import xgboost as xgb
import joblib

# Make the shared project code importable (works whether the notebook is
# opened from the project root or from a subfolder).
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src import data as pipeline
from src import evaluation as ev
from src import forecasters as F
from src import forecasting

DATA_PATH = cfg.find_raw_csv()
MODELS_DIR = cfg.MODELS_DIR
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = cfg.PROCESSED_DIR
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

print("Project root:", ROOT)
print("Raw data:", DATA_PATH.name)

Project root: /tmp/claude-0/-home-claude/7abfd70b-f0e6-5256-8848-f8bb7f3ab4a4/scratchpad/clean
Raw data: wfp_food_prices_uga.csv


## 2. Load and Inspect Raw Data

In [2]:
df_raw = pipeline.load_raw(DATA_PATH)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()

Shape: (34328, 16)
Columns: ['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice']


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,15/01/2006,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,298.40,0.17
1,15/01/2006,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,345.42,0.20
2,15/01/2006,Kampala,Central Kampala,Owino,258,0.32,32.57,cereals and tubers,Rice,52,KG,actual,Wholesale,UGX,1013.40,0.58
3,15/01/2006,Lira,Lira Municipality,Lira,263,2.23,32.91,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,247.76,0.14
4,15/02/2006,Busia,Samia-bugwe,Busia,2665,0.47,34.09,cereals and tubers,Maize,51,KG,actual,Wholesale,UGX,296.59,0.17


In [3]:
print("Missing values:")
print(df_raw.isna().sum())
print("\nPrice stats:")
print(df_raw[["price", "usdprice"]].describe())
print("\nCommodities sample:")
print(df_raw["commodity"].value_counts().head(15))


Missing values:
date            0
admin1          0
admin2          0
market          0
market_id       0
latitude        0
longitude       0
category        0
commodity       0
commodity_id    0
unit            0
priceflag       0
pricetype       0
currency        0
price           0
usdprice        0
dtype: int64

Price stats:
              price      usdprice
count  34328.000000  34328.000000
mean    3188.824076      0.885601
std     3747.237874      1.014938
min        2.000000      0.000570
25%     1200.000000      0.350000
50%     2000.000000      0.580000
75%     3563.000000      0.980000
max    66608.000000     18.150000

Commodities sample:
commodity
Beans               3458
Maize flour         3216
Maize (white)       3068
Sorghum             2942
Oil (vegetable)     2783
Cassava flour       2320
Millet              1790
Salt                 895
Millet flour         890
Cassava (fresh)      838
Leafy vegetables     835
Fish (dry)           833
Milk (fresh)         819
Laundry

## 3. Data Cleaning & Preparation

All rules live in `src/config.py` and are applied by `src/data.py`, so the notebook and the app clean the data the same way.

- Parse dates (DD/MM/YYYY) and store them as the first day of the month
- Keep the six target crops only (fresh cassava is out of scope)
- Keep actual prices recorded in UGX per KG
- Remove the 13 refugee settlement markets
- Map districts to the four UBOS regions (Soroti is Eastern, Kiryandongo is Western)
- Resolve the two maize series: "Maize (white)" is the main series and plain "Maize" only fills months it is missing, because the two disagree by about 23% where they overlap

In [4]:
df, cleaning_log = pipeline.clean_prices(df_raw)

# Calendar helpers used by the EDA cells below
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

display(cleaning_log.to_frame())
print(f"Cleaned shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Markets: {df['market'].nunique()}")
print("\nCommodities:")
print(df["commodity"].value_counts())
print("\nRows per region:")
print(df["region"].value_counts())
df.head()

,step,rows_remaining,rows_removed
0,raw rows,34328,0
1,valid dates,34328,0
2,target crops only,17530,16798
3,actual prices only,17530,0
4,UGX per KG only,17530,0
5,refugee settlement markets removed,13966,3564
6,positive prices only,13966,0
7,one row per series-month,13820,146


Cleaned shape: (13820, 16)
Date range: 2006-01-01 to 2026-04-01
Markets: 30

Commodities:
commodity
Maize            2763
Beans            2563
Maize flour      2322
Cassava flour    2320
Sorghum          2062
Millet           1790
Name: count, dtype: int64

Rows per region:
region
Northern    5413
Eastern     3709
Central     2459
Western     2239
Name: count, dtype: int64


,date,period,commodity,source_commodity,market,market_id,admin1,admin2,region,latitude,longitude,pricetype,price,usdprice,year,month
0,2021-07-01,2021-07,Beans,Beans,Arua,6148,Arua,Arua Municipality,Northern,3.02,30.91,Retail,3633.0,1.03,2021,7
1,2021-10-01,2021-10,Beans,Beans,Arua,6148,Arua,Arua Municipality,Northern,3.02,30.91,Retail,3550.0,0.98,2021,10
2,2022-03-01,2022-03,Beans,Beans,Arua,6148,Arua,Arua Municipality,Northern,3.02,30.91,Retail,3250.0,0.91,2022,3
3,2022-05-01,2022-05,Beans,Beans,Arua,6148,Arua,Arua Municipality,Northern,3.02,30.91,Retail,3500.0,0.97,2022,5
4,2022-06-01,2022-06,Beans,Beans,Arua,6148,Arua,Arua Municipality,Northern,3.02,30.91,Retail,4000.0,1.07,2022,6


## 4. Exploratory Data Analysis

In [5]:
# National average price trends
avg = (df.groupby(["date", "commodity", "pricetype"])["price"]
       .mean().reset_index())

fig = px.line(avg, x="date", y="price", color="commodity",
              facet_col="pricetype", facet_col_wrap=1,
              title="Average Monthly Prices by Commodity (UGX)",
              labels={"price": "Price (UGX)"})
fig.update_layout(height=700)
fig.show()


In [6]:
# Seasonality
monthly_avg = (df[df["pricetype"] == "Retail"]
               .groupby(["commodity", "month"])["price"]
               .mean().reset_index())

fig = px.line(monthly_avg, x="month", y="price", color="commodity",
              markers=True,
              title="Average Retail Price by Calendar Month (Seasonality)")
fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()


In [7]:
# Coverage
coverage = (df.groupby(["commodity", "market"]).size()
            .reset_index(name="n_obs")
            .sort_values("n_obs", ascending=False))
print("Top 15 commodity-market series:")
coverage.head(15)


Top 15 commodity-market series:


,commodity,market,n_obs
84,Maize,Owino,378
25,Beans,Owino,372
73,Maize,Lira,336
14,Beans,Lira,323
54,Cassava flour,Owino,306
43,Cassava flour,Lira,275
91,Maize flour,Gulu,185
107,Maize flour,Mbarara,182
102,Maize flour,Lira,181
113,Maize flour,Owino,181


## 5. Build Monthly Modelling Dataset

Each retail series is placed on a complete monthly calendar, so `shift(1)` always means one month back and `shift(12)` always means the same month last year.

- Gaps of up to 2 months are filled by interpolation (linear in log price) and marked `is_imputed`
- Longer gaps stay missing (`in_long_gap`) and start a new `segment_id`, so lag features become NaN across them and those rows drop out of training
- `price_observed` keeps the raw value, NaN wherever WFP recorded nothing
- `is_outlier_flag` marks one-month spikes that reverse straight away, for review (nothing is removed)

In [8]:
monthly = pipeline.build_monthly_panel(df)          # retail only
series_summary = pipeline.summarise_series(monthly)

print(f"Monthly panel shape: {monthly.shape}")
print(f"Series: {len(series_summary)}  |  reference month: {series_summary.attrs['reference_month']}")
print(f"Observed months: {int(monthly['price_observed'].notna().sum()):,}  "
      f"| interpolated: {int(monthly['is_imputed'].sum()):,}  "
      f"| inside long gaps: {int(monthly['in_long_gap'].sum()):,}")
print(f"Stale series (> {cfg.STALE_AFTER_MONTHS} months old): {int(series_summary['is_stale'].sum())}")
print("\nRetail markets per commodity:")
print(monthly.groupby("commodity")["market"].nunique())
series_summary.sort_values("n_observed", ascending=False).head(10)

Monthly panel shape: (14590, 18)
Series: 154  |  reference month: 2026-04
Observed months: 12,336  | interpolated: 583  | inside long gaps: 1,671
Stale series (> 6 months old): 26

Retail markets per commodity:
commodity
Beans            27
Cassava flour    25
Maize            27
Maize flour      28
Millet           22
Sorghum          25
Name: market, dtype: int64


,commodity,market,pricetype,region,first_month,last_observed_month,span_months,n_observed,n_imputed,n_long_gap_months,n_segments,latest_segment_months,coverage_pct,n_outlier_flags,last_price,months_since_last_obs,is_stale
23,Beans,Owino,Retail,Central,2008-10-01,2026-04-01,211,200,3,8,2,4,94.8,1,4514.0,0,False
48,Cassava flour,Owino,Retail,Central,2008-07-01,2026-04-01,214,195,1,18,3,4,91.1,2,1986.0,0,False
64,Maize,Lira,Retail,Northern,2010-01-01,2025-12-01,192,188,4,0,1,192,97.9,0,920.0,4,False
12,Beans,Lira,Retail,Northern,2010-01-01,2025-12-01,192,186,6,0,1,192,96.9,1,3500.0,4,False
81,Maize flour,Gulu,Retail,Northern,2010-01-01,2026-04-01,196,185,5,6,3,91,94.4,0,2783.0,0,False
97,Maize flour,Mbarara,Retail,Western,2010-01-01,2026-04-01,196,182,11,3,2,91,92.9,0,3000.0,0,False
103,Maize flour,Owino,Retail,Central,2010-01-01,2026-04-01,196,181,4,11,3,4,92.3,0,2720.0,0,False
75,Maize,Owino,Retail,Central,2010-01-01,2025-04-01,184,181,3,0,1,184,98.4,0,2092.0,12,True
92,Maize flour,Lira,Retail,Northern,2010-01-01,2025-12-01,192,181,8,3,2,87,94.3,0,1740.0,4,False
39,Cassava flour,Lira,Retail,Northern,2010-01-01,2025-12-01,192,179,3,10,2,57,93.2,5,2200.0,4,False


## 6. Feature Engineering

In [9]:
FEATURE_COLS = [
    "price_lag1", "price_lag2", "price_lag3", "price_lag6", "price_lag12",
    "roll_mean_3", "roll_mean_6", "roll_std_3",
    "month_sin", "month_cos", "year"
]

def add_time_features(series_df):
    s = series_df.sort_values("date").copy()
    s["price_lag1"] = s["price"].shift(1)
    s["price_lag2"] = s["price"].shift(2)
    s["price_lag3"] = s["price"].shift(3)
    s["price_lag6"] = s["price"].shift(6)
    s["price_lag12"] = s["price"].shift(12)
    s["roll_mean_3"] = s["price"].shift(1).rolling(3).mean()
    s["roll_mean_6"] = s["price"].shift(1).rolling(6).mean()
    s["roll_std_3"] = s["price"].shift(1).rolling(3).std()
    s["month"] = s["date"].dt.month
    s["year"] = s["date"].dt.year
    s["month_sin"] = np.sin(2 * np.pi * s["month"] / 12)
    s["month_cos"] = np.cos(2 * np.pi * s["month"] / 12)
    return s

def prepare_ml_data(series_df):
    fe = add_time_features(series_df)
    fe = fe.dropna(subset=FEATURE_COLS + ["price"])
    return fe[FEATURE_COLS], fe["price"], fe["date"]


## 7. Forecasting Models

### 7.1 Helper functions

In [10]:
def evaluate_forecast(y_true, y_pred, name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"model": name, "MAE": round(mae, 1), "RMSE": round(rmse, 1), "MAPE (%)": round(mape, 2)}

def train_xgboost(X_train, y_train, X_val=None, y_val=None):
    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1
    )
    if X_val is not None:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    else:
        model.fit(X_train, y_train)
    return model


### 7.2 Walk-forward evaluation

Every model is judged the same way (see `src/evaluation.py`):

- **Rolling origins.** For each month *t*, the model sees only data up to *t*, forecasts *t+1* to *t+6*, and is scored against the price WFP actually recorded. Then *t* moves forward one month.
- **Shared calendar cutoffs.** Validation origins run from `VALIDATION_START` to `TEST_START` and only count targets before `TEST_START`. Test origins start at `TEST_START`. Tuning and early stopping use validation only.
- **Same pairs for every model.** A (series, origin, horizon) pair is scored only if every model produced a forecast for it.
- **Baselines.** *naive* repeats the latest price. *seasonal_naive* uses the same month last year scaled to the current level. *app_baseline* is the forecast the Streamlit app currently shows.
- **Skill** = 1 − MAE(model) / MAE(naive). Above zero means the model beats "same as last month".
- **Direction accuracy** = share of forecasts that called the price moving up or down correctly, which is what the sell-or-store advice depends on.

The full run with XGBoost takes around 10 to 15 minutes, so by default the cell loads saved results from `processed/backtest/` (created by `python scripts/run_backtest.py --test`). Set `RUN_BACKTEST = True` to recompute here.

In [11]:
RUN_BACKTEST = False   # True recomputes everything (slow); False loads saved results

saved = cfg.BACKTEST_DIR / "forecasts.parquet"
if RUN_BACKTEST or not saved.exists():
    bt = ev.run_and_save(monthly, F.default_forecasters(), splits=("validation", "test"))
    bt_forecasts, bt_summary = bt["forecasts"], bt["summary"]
else:
    bt_forecasts = pd.read_parquet(saved)
    bt_summary = ev.summary(bt_forecasts)

cols = ["split", "model", "horizon", "n_series", "n_pairs", "MAPE",
        "pooled_skill", "median_skill", "pct_beat", "direction_acc"]
display(bt_summary[cols])
print("Pooled skill vs naive (rows: model, columns: horizon in months)")
display(ev.wide_skill(bt_summary))

,split,model,horizon,n_series,n_pairs,MAPE,pooled_skill,median_skill,pct_beat,direction_acc
0,test,app_baseline,1,152,2249,25.654,-1.793,-1.837,7.237,0.543
1,test,app_baseline,2,152,2101,24.341,-0.776,-0.803,15.132,0.573
2,test,app_baseline,3,140,1952,23.655,-0.395,-0.442,25.000,0.565
3,test,app_baseline,4,140,1812,23.716,-0.234,-0.228,32.143,0.556
4,test,app_baseline,5,140,1677,24.720,-0.181,-0.208,29.286,0.526
5,test,app_baseline,6,123,1538,25.665,-0.186,-0.258,26.829,0.491
6,test,naive,1,152,2249,8.424,0.000,0.000,0.000,NaN
7,test,naive,2,152,2101,12.762,0.000,0.000,0.000,NaN
8,test,naive,3,140,1952,16.217,0.000,0.000,0.000,NaN
9,test,naive,4,140,1812,18.854,0.000,0.000,0.000,NaN


Pooled skill vs naive (rows: model, columns: horizon in months)


horizon                        1      2      3      4      5      6
split      model                                                   
test       app_baseline   -1.793 -0.776 -0.395 -0.234 -0.181 -0.186
           naive           0.000  0.000  0.000  0.000  0.000  0.000
           seasonal_naive -1.000 -0.678 -0.567 -0.518 -0.450 -0.405
           xgb_current    -0.524 -0.308 -0.159 -0.067 -0.015  0.012
validation app_baseline   -2.007 -0.880 -0.513 -0.323 -0.267 -0.277
           naive           0.000  0.000  0.000  0.000  0.000  0.000
           seasonal_naive -0.942 -0.729 -0.768 -0.879 -1.018 -1.125
           xgb_current    -0.540 -0.329 -0.289 -0.255 -0.265 -0.265

In [12]:
fig = px.line(bt_summary, x="horizon", y="MAPE", color="model", facet_col="split",
              markers=True, title="Walk-forward MAPE by forecast horizon",
              labels={"horizon": "Months ahead", "MAPE": "MAPE (%)"})
fig.update_xaxes(dtick=1)
fig.show()

fig = px.line(bt_summary[bt_summary["model"] != "naive"], x="horizon", y="pooled_skill",
              color="model", facet_col="split", markers=True,
              title="Skill vs naive (above 0 = better than 'same as last month')",
              labels={"horizon": "Months ahead", "pooled_skill": "Skill"})
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.update_xaxes(dtick=1)
fig.show()

In [13]:
# Per-crop view on the validation split (this is what Phase 3 model selection will use)
by_crop = ev.summary(bt_forecasts[bt_forecasts["split"] == "validation"], by=["commodity"])
display(by_crop.pivot_table(index=["commodity", "model"], columns="horizon",
                            values="pooled_skill").round(3))

horizon                           1      2      3      4      5      6
commodity     model                                                   
Beans         app_baseline   -1.769 -0.622 -0.332 -0.233 -0.165 -0.148
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -0.737 -0.459 -0.513 -0.674 -0.787 -0.867
              xgb_current    -0.545 -0.182 -0.144 -0.220 -0.231 -0.242
Cassava flour app_baseline   -3.397 -1.539 -0.879 -0.548 -0.489 -0.447
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -1.452 -1.243 -1.246 -1.376 -1.504 -1.442
              xgb_current    -0.865 -0.530 -0.423 -0.370 -0.400 -0.330
Maize         app_baseline   -2.237 -1.001 -0.559 -0.285 -0.172 -0.237
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -0.988 -0.724 -0.762 -0.823 -1.023 -1.274
              xgb_current    -0.535 -0.381 -0.342 -0.184 -0.226 -0.254
Maize flour   app_baseline   -2.515 -1.170 -0.651 -0.424 -0.364 -0.372
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -1.014 -0.820 -0.776 -0.848 -0.933 -1.047
              xgb_current    -0.643 -0.459 -0.311 -0.250 -0.244 -0.235
Millet        app_baseline   -0.590 -0.241 -0.083  0.019 -0.010 -0.019
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -0.388 -0.280 -0.283 -0.297 -0.387 -0.358
              xgb_current    -0.183 -0.215 -0.319 -0.343 -0.372 -0.370
Sorghum       app_baseline   -2.477 -1.237 -0.906 -0.573 -0.539 -0.572
              naive           0.000  0.000  0.000  0.000  0.000  0.000
              seasonal_naive -1.760 -1.543 -1.698 -1.704 -1.930 -2.121
              xgb_current    -0.615 -0.464 -0.473 -0.283 -0.245 -0.249

### 7.3 Model selection for the app

`scripts/select_models.py` runs the same step. For each crop it picks the method with the best average **validation** skill over horizons 1 to 6, and only replaces naive if the margin is at least `SELECTION_MIN_SKILL`. It also works out the forecast band from that method's validation errors, then checks on the test split how often real prices fell inside the band (the target is about 80%). The app reads the result from `processed/model_registry.json`.

In [14]:
registry = forecasting.build_registry(bt_forecasts, write=True)
rows = []
for crop, e in registry["crops"].items():
    for h, b in e["horizons"].items():
        rows.append({"crop": crop, "model": e["model"], "horizon": int(h),
                     "validation_MAPE": b["validation_MAPE"],
                     "test_band_coverage": b["test_band_coverage"]})
sel = pd.DataFrame(rows)
display(pd.DataFrame({c: {"model": e["model"], "reason": e["reason"]} for c, e in registry["crops"].items()}).T)
display(sel.pivot_table(index=["crop", "model"], columns="horizon", values="test_band_coverage"))

,model,reason
Maize,naive,no method beat naive on validation; best chall...
Maize flour,naive,no method beat naive on validation; best chall...
Beans,naive,no method beat naive on validation; best chall...
Sorghum,naive,no method beat naive on validation; best chall...
Millet,naive,no method beat naive on validation; best chall...
Cassava flour,naive,no method beat naive on validation; best chall...


,horizon,1,2,3,4,5,6
crop,model,,,,,,
Beans,naive,0.880,0.878,0.883,0.848,0.839,0.839
Cassava flour,naive,0.774,0.783,0.772,0.740,0.700,0.718
Maize,naive,0.845,0.843,0.814,0.852,0.833,0.851
Maize flour,naive,0.876,0.885,0.956,0.918,0.965,0.966
Millet,naive,0.789,0.726,0.694,0.630,0.544,0.579
Sorghum,naive,0.766,0.713,0.700,0.721,0.728,0.715


### 7.4 Fit XGBoost on all data (kept for continuity)

These models use the pre-Phase-3 set-up and are saved only so the `models/` folder stays populated. The app does not load them yet; Phase 3 replaces this set-up and Phase 4 connects the selected models to the app.

In [15]:
def get_top_series(monthly_df, commodity, n=3):
    subset = monthly_df[(monthly_df["commodity"] == commodity) & monthly_df["price_observed"].notna()]
    return subset.groupby("market").size().sort_values(ascending=False).head(n).index.tolist()

trained_models = {}
for crop in cfg.TARGET_CROPS:
    for mkt in get_top_series(monthly, crop):
        s = pipeline.get_series(monthly, crop, mkt)
        feats = F.xgb_features(s["price"], s["date"]).assign(y=s["price"].to_numpy()).dropna()
        model = xgb.XGBRegressor(**F.XGB_PARAMS).fit(feats[F.XGB_FEATURES], feats["y"])
        fname = f"xgb_{crop.lower().replace(' ', '_')}_{mkt.lower().replace(' ', '_')}_retail.joblib"
        joblib.dump(model, MODELS_DIR / fname)
        trained_models[f"{crop}_{mkt}"] = model
print(f"Saved {len(trained_models)} models to {MODELS_DIR}")

Saved 18 models to /tmp/claude-0/-home-claude/7abfd70b-f0e6-5256-8848-f8bb7f3ab4a4/scratchpad/clean/models


### 7.5 Quick SARIMA demo on one series

Single 12-month hold-out, kept as a demo. Phase 3 adds SARIMA to the walk-forward backtest so it is compared on the same pairs as the other models.


In [16]:
# Pick a long Maize Retail series
demo_market = None
for mkt in ["Owino", "Lira", "Gulu", "Mbarara", "Busia"]:
    cand = monthly[(monthly["commodity"] == "Maize") &
                   (monthly["market"] == mkt) &
                   (monthly["pricetype"] == "Retail")]
    if len(cand) >= 48:
        demo_market = mkt
        break

if demo_market:
    print(f"SARIMA demo: Maize @ {demo_market}")
    demo = (monthly[(monthly["commodity"] == "Maize") &
                    (monthly["market"] == demo_market) &
                    (monthly["pricetype"] == "Retail")]
            .set_index("date")["price"].asfreq("MS").ffill().dropna())

    train, test = demo.iloc[:-12], demo.iloc[-12:]
    try:
        model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
                        enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        pred = res.get_forecast(steps=12)
        metrics = evaluate_forecast(test.values, pred.predicted_mean.values, "SARIMA")
        print(metrics)

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=train.index, y=train.values, name="Train"))
        fig.add_trace(go.Scatter(x=test.index, y=test.values, name="Actual"))
        fig.add_trace(go.Scatter(x=test.index, y=pred.predicted_mean.values,
                                 name="SARIMA", line=dict(dash="dash")))
        fig.update_layout(title=f"SARIMA – Maize @ {demo_market}",
                          xaxis_title="Date", yaxis_title="Price (UGX)", height=400)
        fig.show()
    except Exception as e:
        print(f"SARIMA failed: {e}")
else:
    print("No suitable series found for SARIMA demo.")


SARIMA demo: Maize @ Owino


{'model': 'SARIMA', 'MAE': 248.2, 'RMSE': np.float64(261.0), 'MAPE (%)': 13.07}


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


## 8. Sell-or-Store Advice and Market Comparison

All formulas live in `src/decision.py` and are documented at the top of that file.

- **Timing.** For each option (sell now, or store 1 to 6 months) the expected earnings are forecast price × kg left after storage losses, minus storage fees. Default costs per crop are in `config.CROP_COSTS`, with sources.
- **Chance that storing pays.** The future price is treated as log-normal around the forecast, with spread taken from the forecast band. Storing *k* months pays when the price beats the break-even price *(p₀·q + storage)/(kg left)*.
- **Rule.** Store only if the best storage option also has at least a `STORE_MIN_PROBABILITY` (60%) chance of beating selling now. Otherwise sell now.
- **Confidence.** For storing, the chance it beats selling now. For selling now, one minus the best storage option's chance.
- **Market comparison.** Only markets whose latest price is from the same month as the farmer's market. Road distance is haversine × 1.3, and transport costs `TRANSPORT_UGX_PER_KG_KM` per kg per km.


In [17]:
from src import decision

markets_tbl = pipeline.load_markets()
registry = forecasting.load_registry()
examples = [("Beans", "Lira"), ("Maize", "Owino"), ("Millet", "Mbarara")]

for crop, mkt in examples:
    fc = forecasting.forecast(monthly, crop, mkt, 6, registry)
    costs = decision.Costs.default_for(crop)
    adv = decision.advise(fc, qty=500, costs=costs)
    mc = decision.compare_markets(monthly, markets_tbl, crop, mkt, 500, 6, costs, registry)
    alt = decision.best_alternative(mc)
    print(f"{crop} @ {mkt} (latest price {fc.origin_price:,.0f} UGX/kg, {fc.origin_date:%b %Y})")
    print(f"  advice: {adv.recommendation}   confidence: {adv.confidence_label} "
          f"({adv.confidence:.0%})   best storage option pays off with p = {adv.best_store_prob:.0%}")
    if alt is not None:
        print(f"  best other market: {alt['market']} ({alt['road_km']:.0f} km), "
              f"{alt['vs_home_sell_now_ugx']:+,.0f} UGX vs selling now at home; "
              f"{len(mc.table) - 1} comparable markets, {len(mc.excluded)} left out")
    else:
        print(f"  no other market has a price from {mc.reference_month:%b %Y}")

display(adv.options.round(2))


Beans @ Lira (latest price 3,500 UGX/kg, Dec 2025)
  advice: Sell now   confidence: Low: a close call (56%)   best storage option pays off with p = 44%
  best other market: Kasese (507 km), -10,500 UGX vs selling now at home; 7 comparable markets, 19 left out
Maize @ Owino (latest price 2,092 UGX/kg, Apr 2025)
  advice: Sell now   confidence: Low: a close call (54%)   best storage option pays off with p = 46%
  best other market: Mukono (27 km), +64,000 UGX vs selling now at home; 3 comparable markets, 23 left out


Millet @ Mbarara (latest price 4,011 UGX/kg, Dec 2025)
  advice: Sell now   confidence: Low: a close call (52%)   best storage option pays off with p = 48%
  best other market: Fort Portal (191 km), +458,000 UGX vs selling now at home; 12 comparable markets, 9 left out


,action,months,price,low,high,kg_sold,storage_ugx,transport_ugx,net_ugx,gain_vs_now_ugx,breakeven_price,prob_beats_now
0,Sell now,0,4011.0,4011.0,4011.0,500.00,0.0,0.0,2005500.00,0.00,NaN,NaN
1,Store 1 month,1,4011.0,3469.0,4638.0,499.50,10000.0,0.0,1993494.50,-12005.50,4035.04,0.48
2,Store 2 months,2,4011.0,3341.0,4816.0,499.00,20000.0,0.0,1981491.01,-24008.99,4059.11,0.47
3,Store 3 months,3,4011.0,3195.0,5035.0,498.50,30000.0,0.0,1969489.51,-36010.49,4083.24,0.46
4,Store 4 months,4,4011.0,3209.0,5014.0,498.00,40000.0,0.0,1957490.02,-48009.98,4107.40,0.45
5,Store 5 months,5,4011.0,3327.0,4836.0,497.50,50000.0,0.0,1945492.53,-60007.47,4131.62,0.42
6,Store 6 months,6,4011.0,3170.0,5076.0,497.01,60000.0,0.0,1933497.04,-72002.96,4155.87,0.42


## 9. Export Data for the UI

The build script writes the same files, so `python scripts/build_dataset.py` is enough when only the data changes.

In [18]:
outputs = pipeline.build_all(DATA_PATH, write=True)

print("Exported files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")

print(f"\nModels directory: {len(list(MODELS_DIR.glob('*.joblib')))} model files")
print("\nCore pipeline complete. Proceed to the Streamlit UI.")

Exported files:
  cleaned_prices.parquet  (61.8 KB)
  latest_prices.csv  (8.7 KB)
  markets.csv  (1.2 KB)
  model_registry.json  (13.3 KB)
  monthly_prices.parquet  (144.2 KB)
  series_summary.csv  (14.0 KB)

Models directory: 18 model files

Core pipeline complete. Proceed to the Streamlit UI.


## 10. Evaluating the Advice on Past Prices (Objective 8)

`src/advice_eval.py` replays every past month at every market as if a farmer with 1,000 kg asked for advice. The system sees only prices up to that month. The advice is then scored with the prices that actually followed and compared with:

- **harvest_sale**: sell everything at once at the farmer's market (the Objective 8 baseline)
- **timing_advice**: follow the sell-now-or-store advice
- **full_advice**: also go to the best alternative market when the dashboard shows it pays more after transport. Moves are scored at that market's price one month later, because the farmer learns the price from published data and then has to travel.
- **store_3 / store_6**: always store 3 or 6 months
- **hindsight**: the best timing in hindsight (an upper bound, not achievable)

Harvest months follow the two Ugandan seasons (`config.HARVEST_MONTHS`). Costs are the app's defaults. **Caveat:** WFP prices are retail prices, so gains from moving to another market are an upper bound on what a farmer selling in bulk would get.

Run `python scripts/evaluate_advice.py` to regenerate the results (about 2 minutes); this section loads them.

In [19]:
from src import advice_eval as AE

eval_dir = cfg.ADVICE_EVAL_DIR
if not (eval_dir / "decisions.parquet").exists():
    AE.run_and_save(monthly, pipeline.load_markets(), forecasting.load_registry())
decisions = pd.read_parquet(eval_dir / "decisions.parquet")

cols = ["crop", "decisions", "advised_store_pct", "advised_other_market_pct", "avg_harvest_sale_ugx",
        "timing_advice_gain_pct", "full_advice_gain_pct", "full_advice_better_pct", "full_advice_worse_pct",
        "store_3_gain_pct", "store_6_gain_pct", "hindsight_gain_pct"]
print("Harvest months, per 1,000 kg. Gains are % of the harvest-sale earnings.")
display(AE.summarise(decisions[decisions["is_harvest_month"]], ["crop"])[cols])
display(AE.summarise(decisions[decisions["is_harvest_month"]])[cols[1:]])

Harvest months, per 1,000 kg. Gains are % of the harvest-sale earnings.


,crop,decisions,advised_store_pct,advised_other_market_pct,avg_harvest_sale_ugx,timing_advice_gain_pct,full_advice_gain_pct,full_advice_better_pct,full_advice_worse_pct,store_3_gain_pct,store_6_gain_pct,hindsight_gain_pct
0,Beans,800,0.0,56.6,3293537,0.0,9.41,44.9,11.1,-2.24,-8.86,10.35
1,Maize,776,0.0,38.7,1315624,0.0,3.05,24.7,13.0,-10.54,-14.39,11.61
2,Millet,716,0.0,60.3,2619770,0.0,10.92,50.7,8.6,-2.03,1.60,19.35
3,Sorghum,857,0.0,64.9,1646790,0.0,37.31,60.0,4.0,-4.52,-5.99,11.79


,decisions,advised_store_pct,advised_other_market_pct,avg_harvest_sale_ugx,timing_advice_gain_pct,full_advice_gain_pct,full_advice_better_pct,full_advice_worse_pct,store_3_gain_pct,store_6_gain_pct,hindsight_gain_pct
0,3149,0.0,55.3,2204766,0.0,14.54,45.3,9.0,-3.88,-6.44,13.26


In [20]:
harvest = AE.summarise(decisions[decisions["is_harvest_month"]], ["crop"])
plot_df = harvest.melt(id_vars="crop",
                       value_vars=["timing_advice_gain_pct", "full_advice_gain_pct",
                                   "store_3_gain_pct", "store_6_gain_pct", "hindsight_gain_pct"],
                       var_name="strategy", value_name="gain_pct")
plot_df["strategy"] = plot_df["strategy"].str.replace("_gain_pct", "")
fig = px.bar(plot_df, x="crop", y="gain_pct", color="strategy", barmode="group",
             title="Average gain over selling at harvest (% of harvest-sale earnings, harvest months)",
             labels={"gain_pct": "Gain (%)", "crop": ""})
fig.add_hline(y=0, line_color="grey")
fig.show()

In [21]:
print("Confidence check: stated confidence vs how often the timing advice was right")
display(pd.read_csv(eval_dir / "calibration.csv"))
print("Sensitivity of the main result")
display(pd.read_csv(eval_dir / "sensitivity.csv"))
print("Worst cases from following the full advice (harvest months)")
display(pd.read_csv(eval_dir / "worst_cases.csv"))

Confidence check: stated confidence vs how often the timing advice was right


,band,decisions,stated_confidence_pct,actually_right_pct
0,"[0.0, 0.55)",2587,53.6,40.8
1,"[0.55, 0.6)",6814,57.3,40.4
2,"[0.6, 0.75)",1025,62.0,32.7
3,"[0.75, 1.0)",8,81.6,12.5


Sensitivity of the main result


,scenario,decisions,advised_other_market_pct,full_advice_gain_pct,full_advice_better_pct,full_advice_worse_pct
0,"main (transport 3 UGX/kg/km, 1-month delay, ju...",3149,55.3,14.54,45.3,9.0
1,without the sudden-jump guard,3149,56.2,14.57,45.7,9.5
2,transport cost doubled,3149,31.8,6.56,24.9,6.0
3,no delay before selling at another market,3149,55.3,19.19,55.3,0.0


Worst cases from following the full advice (harvest months)


,crop,market,decision_month,advice,advice_market,alt_road_km,price_now,harvest_sale,full_advice,gain_ugx,gain_pct
0,Millet,Kotido,2024-01-01,Sell now,Hoima,459.0,3600.0,3600000.0,1023000.0,-2577000.0,-71.6
1,Millet,Mbarara,2024-01-01,Sell now,Hoima,312.0,4000.0,4000000.0,1464000.0,-2536000.0,-63.4
2,Millet,Masaka,2024-01-01,Sell now,Hoima,263.0,4100.0,4100000.0,1611000.0,-2489000.0,-60.7
3,Millet,Soroti,2024-01-01,Sell now,Hoima,331.0,3800.0,3800000.0,1407000.0,-2393000.0,-63.0
4,Millet,Kasese,2024-01-01,Sell now,Hoima,260.0,4000.0,4000000.0,1620000.0,-2380000.0,-59.5
5,Millet,Wakiso,2024-01-01,Sell now,Hoima,221.0,4000.0,4000000.0,1737000.0,-2263000.0,-56.6
6,Millet,Makaratin,2024-01-01,Sell now,Hoima,379.0,3500.0,3500000.0,1263000.0,-2237000.0,-63.9
7,Beans,Gulu,2024-12-01,Sell now,Lira,117.0,4000.0,4000000.0,1769000.0,-2231000.0,-55.8
8,Millet,Mukono,2024-01-01,Sell now,Hoima,255.0,3825.0,3825000.0,1635000.0,-2190000.0,-57.3
9,Millet,Jinja,2024-01-01,Sell now,Hoima,305.0,3575.0,3575000.0,1485000.0,-2090000.0,-58.5


## Status and Next Steps

**Done**
- Clean, gap-aware monthly panel with shared rules (`src/data.py`, `src/config.py`)
- Walk-forward evaluation against naive and seasonal-naive baselines at horizons 1 to 6 (`src/evaluation.py`)
- Per-crop model selection on the validation split, with forecast bands checked on the test split (`src/forecasting.py`)
- Sell-or-store and market comparison with crop-specific costs, real distances and probability-based confidence (`src/decision.py`)
- Advice tested on past prices, Objective 8 (`src/advice_eval.py`)
- AI explanations and questions with a number check, services platform for Part 2

**Next (Phase 3, model improvement)**
1. Predict the month-on-month log change instead of the price level, and drop `year`
2. Direct multi-horizon models, and one pooled model across markets with market and region features
3. Add SARIMA and LSTM to the walk-forward backtest so all three families are compared on the same pairs
4. Tune hyperparameters on the validation split only
5. Model the upward price drift and recalibrate the bands; Section 10 shows the stated confidence is too high (about 55% stated, 33 to 41% realised)
6. Report the test split once, after these choices are final

**Known limits**
- WFP prices are retail, so gains from moving to another market are an upper bound for a farmer selling in bulk
- Storage fees and transport rates are assumptions the farmer can change; no published Ugandan tariff was found
- No SMS or email notifications, payments or document upload in the services platform yet
